# NYT Games Vanilla LLM Benchmark (Self-Contained)

This notebook is **self-contained** for Colab.  
You can click **Run all** without needing any extra Python files.

It supports:
- **Hugging Face local model** evaluation with **no training**
- **Cloud LLM** evaluation through an **OpenAI-compatible API** (including Triton-style setup)

Choose the backend in the config cell below:
- `BACKEND = "hf"`
- `BACKEND = "cloud"`


In [ ]:
# Colab / notebook setup
!pip -q install transformers accelerate openai pandas sentencepiece

In [ ]:
# =========================
# Configuration
# =========================

# Choose one:
BACKEND = "cloud"        # "hf" or "cloud"

# Games: "spelling_bee", "wordle", or "both"
GAME = "both"

# Number of puzzles to evaluate
NUM_PUZZLES = 100

# Output directory
OUTPUT_DIR = "./benchmark_outputs"

# Prompt / decoding settings
USE_THINKING_FORMAT = False
USE_BEAM_SEARCH = False
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 32
NUM_BEAMS = 4

# -------------------------
# Hugging Face backend
# -------------------------
HF_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_DTYPE = "auto"    # "auto", "float16", "bfloat16", "float32"

# -------------------------
# Cloud backend
# -------------------------
# Matches the Triton-style setup pattern from your notebook.
CLOUD_MODEL_NAME = "api-gpt-oss-120b"
CLOUD_BASE_URL = "https://tritonai-api.ucsd.edu"

# Put your key in the environment before running this cell, or set it here.
# Example:
import os
os.environ["OPENAI_API_KEY"] = "sk-vCGngQGFwVOFYFaF0mldhg"


In [ ]:
# Imports
from __future__ import annotations

import json
import math
import os
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import pandas as pd

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, Protocol

import pandas as pd


def ensure_repo(repo_dir: str = "./cse291-nytgames",
                repo_url: str = "https://github.com/jackiepiepkorn/cse291-nytgames.git") -> Path:
    repo_path = Path(repo_dir).resolve()
    if not repo_path.exists():
        print(f"Cloning benchmark repo into {repo_path} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    return repo_path


# Late imports after repo is present.
def load_game_modules(repo_dir: str = "./cse291-nytgames"):
    ensure_repo(repo_dir)
    from nytgames import SpellingBeeConfig, SpellingBeeEnv, SpellingBeeDataset, WordleConfig, WordleEnv, load_dictionary
    from nytgames.env.wordle import _score_guess, _format_feedback
    from nytgames.data.dataset import _SPELLING_BEE_CSV_PATH as SB_CSV_PATH, _PROMPTS_DIR
    return {
        "SpellingBeeConfig": SpellingBeeConfig,
        "SpellingBeeEnv": SpellingBeeEnv,
        "SpellingBeeDataset": SpellingBeeDataset,
        "WordleConfig": WordleConfig,
        "WordleEnv": WordleEnv,
        "load_dictionary": load_dictionary,
        "_score_guess": _score_guess,
        "_format_feedback": _format_feedback,
        "SB_CSV_PATH": SB_CSV_PATH,
        "_PROMPTS_DIR": _PROMPTS_DIR,
    }


SB_SYSTEM_PROMPT_THINKING = """You are an expert NYT Spelling Bee player. In this game you must guess words that:
  1. Are at least 4 letters long.
  2. Use only the allowed letters (letters may be reused).
  3. Always contain the center letter.
A pangram uses every allowed letter at least once and earns a bonus.

Think step by step, then give your answer.
Format your response as:
Reasoning:
- (your reasoning about which letters to use)
Answer:
(single word guess)"""

WORDLE_SYSTEM_PROMPT_THINKING = """You are an expert Wordle player. In this game you must guess a secret 5-letter word in 6 tries.
After each guess you receive feedback for every letter:
  - GREEN: the letter is correct and in the right position.
  - YELLOW: the letter is in the word but in the wrong position.
  - GRAY: the letter is not in the word at all.
Use the feedback to narrow down possibilities.

Think step by step, then give your answer.
Format your response as:
Reasoning:
- (your reasoning about the feedback)
Answer:
(single 5-letter word guess)"""


class GuessBackend(Protocol):
    def generate_text(self, messages: list[dict[str, str]], *, max_new_tokens: int = 32,
                      temperature: float = 0.7, use_beam_search: bool = False,
                      num_beams: int = 1) -> str: ...

    def score_candidates(self, messages: list[dict[str, str]], candidates: list[str]) -> list[float] | None: ...


@dataclass
class BenchmarkContext:
    use_thinking_format: bool = True
    use_beam_search_eval: bool = True
    eval_num_beams: int = 8
    score_top_k: int = 30
    verbose: bool = True


class CandidateCaches:
    def __init__(self, load_dictionary: Callable, score_guess: Callable):
        self.full_dictionary = load_dictionary()
        self.wordle_word_set = load_dictionary(length=5)
        self.word_bitmask_cache = {w: self._letter_bitmask(w) for w in self.full_dictionary}
        self.score_guess = score_guess

    @staticmethod
    def _letter_bitmask(word: str) -> int:
        mask = 0
        for ch in word.upper():
            if "A" <= ch <= "Z":
                mask |= 1 << (ord(ch) - ord("A"))
        return mask

    def get_sb_candidates(self, allowed_letters: set[str], center_letter: str, min_len: int = 4) -> list[str]:
        allowed_upper = {l.upper() for l in allowed_letters}
        center_upper = center_letter.upper()
        allowed_mask = 0
        for l in allowed_upper:
            allowed_mask |= 1 << (ord(l) - ord("A"))
        center_bit = 1 << (ord(center_upper) - ord("A"))

        candidates: list[str] = []
        for word, wmask in self.word_bitmask_cache.items():
            if len(word) >= min_len and (wmask & ~allowed_mask) == 0 and (wmask & center_bit):
                candidates.append(word)
        return candidates


def extract_answer(text: str) -> str:
    m = re.search(r"(?:Answer|ANSWER|answer)\s*:\s*([A-Za-z]+)", text)
    if m:
        return m.group(1).strip()
    clean = text.strip()
    lines = [l.strip() for l in clean.split("\n") if l.strip()]
    if lines:
        last = lines[-1]
        words = re.findall(r"[A-Za-z]+", last)
        if words:
            return words[-1]
    return clean


def clean_alpha_word(text: str, limit: int | None = None) -> str:
    word = "".join(ch for ch in extract_answer(text) if ch.isalpha()).upper()
    if limit is not None:
        word = word[:limit]
    return word


def generate_guess(backend: GuessBackend, messages: list[dict[str, str]], ctx: BenchmarkContext,
                   *, temperature: float = 0.7) -> str:
    raw = backend.generate_text(
        messages,
        max_new_tokens=64 if ctx.use_thinking_format else 32,
        temperature=temperature,
        use_beam_search=ctx.use_beam_search_eval,
        num_beams=ctx.eval_num_beams,
    )
    return extract_answer(raw)


def generate_from_candidates(backend: GuessBackend, messages: list[dict[str, str]], candidates: list[str],
                             ctx: BenchmarkContext, *, temperature: float = 0.0) -> str:
    if not candidates:
        return generate_guess(backend, messages, ctx, temperature=max(temperature, 0.7))

    sample = list(candidates)
    if len(sample) > ctx.score_top_k:
        sample = random.sample(sample, ctx.score_top_k)

    scores = backend.score_candidates(messages, sample)
    if scores is None:
        # Fallback for cloud APIs that cannot expose token scores.
        shortlist = sample[: min(5, len(sample))]
        user_prompt = (
            "Choose exactly one best next guess from this candidate list and output only the word.\n"
            f"Candidates: {', '.join(shortlist)}"
        )
        augmented = list(messages) + [{"role": "user", "content": user_prompt}]
        chosen = clean_alpha_word(
            backend.generate_text(
                augmented,
                max_new_tokens=16,
                temperature=0.0,
                use_beam_search=False,
                num_beams=1,
            )
        )
        return chosen if chosen in set(shortlist) else shortlist[0]

    best_idx = max(range(len(sample)), key=lambda i: scores[i])
    return sample[best_idx]


def evaluate_spelling_bee(backend: GuessBackend, ctx: BenchmarkContext, *, repo_dir: str,
                          csv_path: str | None = None, num_puzzles: int = 10,
                          max_guesses: int = 10) -> pd.DataFrame:
    mods = load_game_modules(repo_dir)
    SpellingBeeDataset = mods["SpellingBeeDataset"]
    SpellingBeeEnv = mods["SpellingBeeEnv"]
    SB_CSV_PATH = mods["SB_CSV_PATH"]
    load_dictionary = mods["load_dictionary"]
    caches = CandidateCaches(load_dictionary, mods["_score_guess"])

    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH, max_guesses=max_guesses)
    base_system_prompt = SB_SYSTEM_PROMPT_THINKING if ctx.use_thinking_format else SpellingBeeDataset._SYSTEM_PROMPT
    results: list[dict] = []

    for idx in range(min(num_puzzles, len(sb_ds))):
        item = sb_ds[idx]
        config = sb_ds.get_config(item["puzzle_id"], max_guesses=max_guesses)
        env = SpellingBeeEnv(config)
        obs, _ = env.reset()

        messages = list(item["prompt"])
        messages[0] = {"role": "system", "content": base_system_prompt}

        sb_candidates = caches.get_sb_candidates(config.letter_set, config.center_letter)
        already_guessed: set[str] = set()
        allowed_letters = {l.upper() for l in config.letter_set}
        center = config.center_letter.upper()

        if ctx.verbose:
            print(f"[SB] Puzzle {item['puzzle_id']:>3}: letters={''.join(sorted(config.letter_set))}, center={config.center_letter}")

        def is_valid_sb(word: str) -> bool:
            return (
                len(word) >= 4
                and set(word) <= allowed_letters
                and center in word
                and word not in already_guessed
            )

        for _turn in range(max_guesses):
            remaining_candidates = [c for c in sb_candidates if c not in already_guessed]
            if remaining_candidates:
                word = generate_from_candidates(backend, messages, remaining_candidates, ctx).upper()
            else:
                word = ""
                best_fallback = ""
                for attempt in range(10):
                    candidate = clean_alpha_word(generate_guess(backend, messages, ctx, temperature=0.7 + 0.05 * attempt))
                    if is_valid_sb(candidate):
                        word = candidate
                        break
                    if candidate and candidate not in already_guessed and not best_fallback:
                        best_fallback = candidate
                word = word or best_fallback or ""

            if not word:
                break

            already_guessed.add(word)
            messages.append({"role": "assistant", "content": word})
            obs, reward, terminated, truncated, _ = env.step(word)

            if ctx.verbose:
                status = "Correct" if reward > 0 else "Incorrect"
                print(f"  [SB Guess] {word} -> {status} (Points: {reward if reward > 0 else 0})")

            if terminated or truncated:
                break

            if reward > 0:
                fb = f"'{word}' was correct! Running total: {obs['total_points']}."
            else:
                fb = f"'{word}' was not accepted. {obs['feedback']}"
            fb += (
                f"\nREMINDER: Only use letters {', '.join(sorted(allowed_letters))} "
                f"(center {center} required, min 4 letters)."
                f"\nPlease guess another word. Do NOT repeat any previous guess."
            )
            messages.append({"role": "user", "content": fb})

        results.append({
            "benchmark": "spelling_bee",
            "puzzle_id": item["puzzle_id"],
            "letters": "".join(sorted(config.letter_set)),
            "center": config.center_letter,
            "available_words": len(config.word_set),
            "found": len(obs["valid_words_guessed"]),
            "score": obs["total_points"],
            "guess_ratio": len(obs["valid_words_guessed"]) / max(len(config.word_set), 1),
        })
        if ctx.verbose:
            print(
                f"[SB] puzzle={item['puzzle_id']:>3} letters={''.join(sorted(config.letter_set))} "
                f"found={len(obs['valid_words_guessed'])}/{len(config.word_set)} score={obs['total_points']}"
            )

    return pd.DataFrame(results)


def evaluate_wordle(backend: GuessBackend, ctx: BenchmarkContext, *, repo_dir: str,
                    num_puzzles: int = 20, max_guesses: int = 6,
                    solutions_path: str | None = None) -> pd.DataFrame:
    mods = load_game_modules(repo_dir)
    WordleConfig = mods["WordleConfig"]
    WordleEnv = mods["WordleEnv"]
    load_dictionary = mods["load_dictionary"]
    score_guess = mods["_score_guess"]
    format_feedback = mods["_format_feedback"]
    SB_CSV_PATH = mods["SB_CSV_PATH"]
    data_dir = Path(SB_CSV_PATH).parent
    wordle_solutions = Path(solutions_path) if solutions_path else data_dir / "wordle_past_solutions.txt"

    words = [w.strip().upper() for w in wordle_solutions.read_text().splitlines() if w.strip()][:num_puzzles]
    word_set = load_dictionary(length=5)
    system_prompt = WORDLE_SYSTEM_PROMPT_THINKING if ctx.use_thinking_format else (mods["_PROMPTS_DIR"] / "wordle_system.md").read_text().strip()
    user_template = (mods["_PROMPTS_DIR"] / "wordle_user.md").read_text().strip()
    results: list[dict] = []

    for target in words:
        cfg = WordleConfig(target_word=target, word_set=word_set, max_guesses=max_guesses)
        env = WordleEnv(cfg)
        obs, _ = env.reset()

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_template.format(max_guesses=max_guesses)},
        ]

        green: dict[int, str] = {}
        yellow: dict[int, set[str]] = {}
        gray: set[str] = set()
        previous_guesses: set[str] = set()

        if ctx.verbose:
            print(f"[WD] Target Word: {target}")

        for turn_num in range(max_guesses):
            word = ""
            best_fallback = ""
            for attempt in range(10):
                candidate = clean_alpha_word(generate_guess(backend, messages, ctx, temperature=0.7 + 0.05 * attempt), limit=5)
                if len(candidate) == 5 and candidate in word_set and candidate not in previous_guesses:
                    word = candidate
                    break
                if len(candidate) == 5 and candidate not in previous_guesses and not best_fallback:
                    best_fallback = candidate
            if not word:
                remaining = list(word_set - previous_guesses)
                word = best_fallback if best_fallback in word_set else (random.choice(remaining) if remaining else "ARISE")

            previous_guesses.add(word)
            messages.append({"role": "assistant", "content": word})
            obs, reward, terminated, truncated, _ = env.step(word)

            if ctx.verbose:
                status_emoji = "🟩" if terminated and reward > 0 else "❌"
                print(f"  [WD Guess {turn_num + 1}/{max_guesses}] {word} -> {obs['feedback']} {status_emoji}")

            if terminated or truncated:
                break

            if obs["feedback"] and "Invalid" not in obs["feedback"]:
                tiles = score_guess(word, target)
                for pos, (letter, tile) in enumerate(zip(word, tiles)):
                    if tile == "correct":
                        green[pos] = letter
                    elif tile == "present":
                        yellow.setdefault(pos, set()).add(letter)
                    else:
                        gray.add(letter)

            hints: list[str] = []
            if green:
                hints.append("Confirmed: " + ", ".join(f"{l} at position {p+1}" for p, l in sorted(green.items())))
            present = set().union(*yellow.values()) if yellow else set()
            present -= set(green.values())
            if present:
                hints.append(f"In word but position unknown: {', '.join(sorted(present))}")
            truly_gray = gray - set(green.values()) - present
            if truly_gray:
                hints.append(f"Not in word: {', '.join(sorted(truly_gray))}")

            fb_msg = f"Feedback: {obs['feedback']}\nYou have {max_guesses - obs['num_guesses']} attempts remaining."
            if hints:
                fb_msg += "\n\nWhat we know:\n" + "\n".join(hints)
            fb_msg += f"\nPrevious guesses: {', '.join(sorted(previous_guesses))}. Do NOT repeat any of these."
            fb_msg += "\nPlease guess another 5-letter word."
            messages.append({"role": "user", "content": fb_msg})

        won = bool(obs.get("feedback")) and target in previous_guesses
        results.append({
            "benchmark": "wordle",
            "target": target,
            "won": won,
            "guesses_used": obs["num_guesses"],
            "remaining_guesses": max_guesses - obs["num_guesses"],
            "final_feedback": obs["feedback"],
        })
        if ctx.verbose:
            print(f"[WD] target={target} won={won} guesses={obs['num_guesses']} feedback={obs['feedback']}")

    return pd.DataFrame(results)


def summarize_results(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"rows": 0}

    if "benchmark" not in df.columns:
        return {
            "rows": int(len(df)),
            "columns": list(df.columns),
            "warning": "No benchmark column found in results."
        }

    benchmarks = sorted(df["benchmark"].dropna().unique().tolist())

    if len(benchmarks) == 1:
        benchmark = benchmarks[0]
        sub = df[df["benchmark"] == benchmark]

        if benchmark == "spelling_bee":
            out = {
                "benchmark": benchmark,
                "rows": int(len(sub)),
            }
            if "score" in sub.columns:
                out["avg_score"] = float(sub["score"].mean())
                out["max_score"] = float(sub["score"].max())
            if "found" in sub.columns:
                out["avg_found"] = float(sub["found"].mean())
            if "guess_ratio" in sub.columns:
                out["avg_guess_ratio"] = float(sub["guess_ratio"].mean())
            return out

        if benchmark == "wordle":
            out = {
                "benchmark": benchmark,
                "rows": int(len(sub)),
            }
            if "won" in sub.columns:
                out["win_rate"] = float(sub["won"].mean())
            if "guesses_used" in sub.columns:
                out["avg_guesses_used"] = float(sub["guesses_used"].mean())
            return out

        return {
            "benchmark": benchmark,
            "rows": int(len(sub)),
        }

    summary = {
        "benchmarks": benchmarks,
        "rows": int(len(df)),
    }

    per_benchmark = {}
    for benchmark in benchmarks:
        sub = df[df["benchmark"] == benchmark]
        entry = {"rows": int(len(sub))}

        if benchmark == "spelling_bee":
            if "score" in sub.columns:
                entry["avg_score"] = float(sub["score"].mean())
                entry["max_score"] = float(sub["score"].max())
            if "found" in sub.columns:
                entry["avg_found"] = float(sub["found"].mean())
            if "guess_ratio" in sub.columns:
                entry["avg_guess_ratio"] = float(sub["guess_ratio"].mean())

        elif benchmark == "wordle":
            if "won" in sub.columns:
                entry["win_rate"] = float(sub["won"].mean())
            if "guesses_used" in sub.columns:
                entry["avg_guesses_used"] = float(sub["guesses_used"].mean())

        per_benchmark[benchmark] = entry

    summary["per_benchmark"] = per_benchmark
    return summary


def write_outputs(df: pd.DataFrame, out_prefix: str) -> None:
    out_path = Path(out_prefix)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path = out_path.with_suffix(".csv")
    json_path = out_path.with_suffix(".summary.json")
    df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summarize_results(df), f, indent=2)
    print(f"Saved detailed results to {csv_path}")
    print(f"Saved summary to {json_path}")
    print(json.dumps(summarize_results(df), indent=2))



import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI

class HFBackend:
    def __init__(self, model_name: str, dtype: str = "auto"):
        if dtype == "auto":
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
                torch_dtype = torch.bfloat16
            elif torch.cuda.is_available():
                torch_dtype = torch.float16
            else:
                torch_dtype = torch.float32
        else:
            torch_dtype = getattr(torch, dtype)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map="auto",
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.eval()

    def generate_text(self, messages, *, max_new_tokens=32, temperature=0.7,
                      use_beam_search=False, num_beams=1) -> str:
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            if use_beam_search:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    do_sample=False,
                    early_stopping=True,
                )
            else:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.95,
                )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    def score_candidates(self, messages, candidates):
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        scores = []
        with torch.no_grad():
            for word in candidates:
                word_ids = self.tokenizer.encode(word, add_special_tokens=False, return_tensors="pt").to(self.model.device)
                combined = torch.cat([inputs["input_ids"], word_ids], dim=1)
                outputs = self.model(combined)
                logits = outputs.logits[0, inputs["input_ids"].shape[1]-1:-1, :]
                log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
                word_log_prob = sum(log_probs[t, word_ids[0, t]].item() for t in range(word_ids.shape[1]))
                scores.append(word_log_prob)
        return scores

class CloudBackend:
    def __init__(
        self,
        model: str,
        api_key: str | None = None,
        base_url: str | None = None,
    ):
        self.model = model

        # Match the setup pattern from the attached notebook:
        #   api_key = os.environ["OPENAI_API_KEY"]
        #   client = OpenAI(..., base_url="https://tritonai-api.ucsd.edu")
        if api_key is None:
            api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise ValueError(
                "No API key found. Set OPENAI_API_KEY in your environment or pass --api-key."
            )

        if base_url is None:
            base_url = os.environ.get("OPENAI_BASE_URL", "https://tritonai-api.ucsd.edu")

        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.base_url = base_url

    def generate_text(
        self,
        messages,
        *,
        max_new_tokens=32,
        temperature=0.7,
        use_beam_search=False,
        num_beams=1,
    ) -> str:
        # num_beams is ignored for cloud chat APIs.
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_new_tokens,
        )
        return resp.choices[0].message.content or ""

    def score_candidates(self, messages, candidates):
        # Most cloud chat APIs do not expose token logprobs consistently.
        return None

def make_backend():
    if BACKEND == "hf":
        return HFBackend(HF_MODEL_NAME, dtype=HF_DTYPE)
    elif BACKEND == "cloud":
        return CloudBackend(
            model=CLOUD_MODEL_NAME,
            api_key=os.environ.get("OPENAI_API_KEY"),
            base_url=CLOUD_BASE_URL,
        )
    else:
        raise ValueError(f"Unsupported BACKEND: {BACKEND}")

def run_benchmark():
    backend = make_backend()
    ctx = BenchmarkContext(
        use_thinking_format=USE_THINKING_FORMAT,
        use_beam_search_eval=USE_BEAM_SEARCH,
        verbose=True,
    )

    frames = []
    repo_directory = "./cse291-nytgames"

    if GAME in ("spelling_bee", "both"):
        print("\n=== Running Spelling Bee ===")
        sb_results = evaluate_spelling_bee(
            backend,
            ctx,
            num_puzzles=NUM_PUZZLES,
            repo_dir=repo_directory,
        )
        frames.append(sb_results)
        print(f"Spelling Bee finished: {len(sb_results)} rows")

    if GAME in ("wordle", "both"):
        print("\n=== Running Wordle ===")
        wd_results = evaluate_wordle(
            backend,
            ctx,
            num_puzzles=NUM_PUZZLES,
            repo_dir=repo_directory,
        )
        frames.append(wd_results)
        print(f"Wordle finished: {len(wd_results)} rows")

    if len(frames) == 0:
        results_df = pd.DataFrame()
    else:
        results_df = pd.concat(frames, ignore_index=True)

    out_dir = Path(OUTPUT_DIR)
    out_prefix = out_dir / f"{BACKEND}_{GAME}_{NUM_PUZZLES}"
    write_outputs(results_df, str(out_prefix))

    csv_path = out_prefix.with_suffix(".csv")
    json_path = out_prefix.with_suffix(".summary.json")
    with open(json_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    print("\nSaved files:")
    print("CSV :", csv_path)
    print("JSON:", json_path)

    return results_df, summary, csv_path, json_path

In [ ]:
# Run benchmark
results, summary, csv_path, json_path = run_benchmark()


=== Running Spelling Bee ===
[SB] Puzzle   1: letters=AHORTWY, center=W
  [SB Guess] WAWA -> Incorrect (Points: 0)
  [SB Guess] AWORRY -> Incorrect (Points: 0)
  [SB Guess] YAWROOT -> Incorrect (Points: 0)
  [SB Guess] THATAWAY -> Correct (Points: 8)
  [SB Guess] ARROWY -> Incorrect (Points: 0)
  [SB Guess] HOARWORT -> Incorrect (Points: 0)
  [SB Guess] WARATAH -> Incorrect (Points: 0)
  [SB Guess] ROWY -> Incorrect (Points: 0)
  [SB Guess] WAHWAH -> Incorrect (Points: 0)
  [SB Guess] TWAT -> Incorrect (Points: 0)
[SB] puzzle=  1 letters=AHORTWY found=1/26 score=8
[SB] Puzzle   2: letters=CFIMNOR, center=I
  [SB Guess] CINCINNI -> Incorrect (Points: 0)
  [SB Guess] CONCIO -> Incorrect (Points: 0)
  [SB Guess] FRIM -> Incorrect (Points: 0)
  [SB Guess] OFFIC -> Incorrect (Points: 0)
  [SB Guess] RICININIC -> Incorrect (Points: 0)
  [SB Guess] CONI -> Incorrect (Points: 0)
  [SB Guess] CORONION -> Incorrect (Points: 0)
  [SB Guess] MINO -> Incorrect (Points: 0)
  [SB Guess] MICRO -> Cor

AuthenticationError: Error code: 401 - {'error': {'message': 'Authentication Error, Error in connector: Error querying the database: FATAL: the database system is shutting down', 'type': 'auth_error', 'param': 'None', 'code': '401'}}

In [ ]:
# Show summary
summary

In [ ]:
# Show first few result rows
df = pd.read_csv(csv_path)
df.head(20)